### Notebook for verifying that implemented dynamics work correctly

In [ ]:
import importlib
import Model
importlib.reload(Model)
import DeerClass
importlib.reload(DeerClass)
import WolfClass
importlib.reload(WolfClass)
from Model import SpeciesModel
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import solara
from mesa.visualization import SolaraViz, make_space_component
from matplotlib.patches import Circle, Patch
from matplotlib.figure import Figure

In [ ]:
AGENT_COLOURS = {
    "Deer": "orange",
    "Wolf": "blue",
}
SENSING_RADIUS = {
    "Deer": 1.5,
    "Wolf": 2,
}
def agent_draw(agent):
    if agent.species in AGENT_COLOURS:
        return {"color": AGENT_COLOURS[agent.species], "size": 5}
    return {"color": "green", "size": 1}


# @solara.component
def SpaceWithSensing(model):
    if isinstance(model, solara.Reactive):
        model = model.value

    fig = Figure(figsize=(8, 8))
    ax = fig.subplots()

    species_present = set()

    for agent in model.agents:
        if agent.pos is None:
            continue

        color = AGENT_COLOURS.get(agent.species, "green")
        size = 5 if agent.species in AGENT_COLOURS else 1

        # Draw the agent
        ax.scatter(
            agent.pos[0], agent.pos[1],
            c=color, s=size * 10, zorder=3
        )

        # Draw sensing radius circle
        if agent.species in SENSING_RADIUS:
            circle = Circle(
                (agent.pos[0], agent.pos[1]),
                radius=SENSING_RADIUS[agent.species],
                edgecolor=color,
                facecolor=color,
                alpha=0.15,        # Transparent fill
                linewidth=1.5,
                linestyle="--",    # Dashed border
                zorder=2
            )
            ax.add_patch(circle)

        species_present.add(agent.species)

    # Build legend with both agent dot and sensing radius
    legend_elements = []
    for species in AGENT_COLOURS:
        if species in species_present:
            legend_elements.append(
                Patch(
                    facecolor=AGENT_COLOURS[species],
                    alpha=0.3,
                    edgecolor=AGENT_COLOURS[species],
                    label=f"{species} (r={SENSING_RADIUS.get(species, '?')})"
                )
            )

    ax.legend(
        handles=legend_elements,
        loc="upper right",
        fontsize=10,
        framealpha=0.9
    )

    # Style
    ax.set_xlim(0, model.space.x_max)
    ax.set_ylim(0, model.space.y_max)
    ax.set_aspect("equal")
    ax.set_title(f"Step: {model.steps}", fontsize=14)
    ax.set_xlabel("X (km)")
    ax.set_ylabel("Y (km)")
    ax.set_facecolor("#f5f5f0")
    fig.tight_layout()

    solara.FigureMatplotlib(fig)


### Wolf pack movement 
Step size has been decreased to 15 mins

In [ ]:
# Parameters to view wolf pack dynamics and sensing radius
model_params = {
    "max_steps": 1500,
    "init_predators": 10,
    "init_deer": 1,
    "height": 30,
    "width": 30,
    "step_size": 0.25,
    "yearly_sunlight_hours": 8760,
    "seed": None,
    "predator": "Wolf",
    "energy_decrease": 0.002,
    "pack_limit": 12,
    "veg_patch_spacing": 4,
    "sapling_density": 20,
    "tree_density": 8,
    "sapling_regrowth_prob": 1/10,
    "sapling_maturation_prob": 1/20000,
    "use_base": False,
    "use_pack_dynamics": True,
    "use_random_movement": False,
    "use_veg": False,
    "use_boundary_conditions": True,
}

# Create the model
small_world = SpeciesModel(**model_params)

SolaraViz(
    small_world,
    components=[
        # make_space_component(agent_portrayal=agent_draw, backend="matplotlib"),
        SpaceWithSensing
    ],
    model_params=model_params,
    name="Predator-Prey Simulation",
)

In [ ]:
# AGENT_COLOURS = {
#     "Deer": "orange",
#     "Wolf": "blue",
#     "Lynx": "brown",
#     "Sapling": "#a2c399",
#     "Tree": "#5e8354",
# }

# def apply_colours(ax):
#     """Reusable - applies AGENT_COLOURS to any plot"""
#     for line in ax.get_lines():
#         if line.get_label() in AGENT_COLOURS:
#             line.set_color(AGENT_COLOURS[line.get_label()])
#     ax.legend()

# def agent_draw(agent):
#     """Display the agents with assigned colours."""
#     if agent.species == "Deer":
#         return {"color": AGENT_COLOURS["Deer"], "size": 5}
#     elif agent.species == "Wolf":
#         return {"color": AGENT_COLOURS["Wolf"], "size": 5}
#     elif agent.species == "Lynx":
#         return {"color": AGENT_COLOURS["Lynx"], "size": 5} 
#     elif agent.species == "Vegetation":
#         return{"color":AGENT_COLOURS["Tree"], "size": 5}

# def show_in_solara(model):
#     space_component = make_space_component(
#         agent_portrayal=agent_draw,
#         backend="matplotlib"
#     )
#     page = SolaraViz(
#         model, 
#         components=[space_component],
#         # model_params=model_params,
#         name="Species Model",
#     )
#     return page


### Deer Movement

In [ ]:
# Parameters to view deer fleeing sensing radius
model_params = {
    "max_steps": 1500,
    "init_predators": 2,
    "init_deer": 4,
    "height": 30,
    "width": 30,
    "step_size": 0.25,
    "yearly_sunlight_hours": 8760,
    "seed": None,
    "predator": "Wolf",
    "energy_decrease": 0.002,
    "pack_limit": 12,
    "veg_patch_spacing": 4,
    "sapling_density": 20,
    "tree_density": 8,
    "sapling_regrowth_prob": 1/10,
    "sapling_maturation_prob": 1/20000,
    "use_base": False,
    "use_pack_dynamics": True,
    "use_random_movement": False,
    "use_veg": False,
    "given_positions": {
        "Deer": [np.array([5, 5]), np.array([10, 10]), np.array([3, 3]), np.array([9.5, 9.5])],
        "Wolf": [np.array([4, 4]), np.array([11, 11])]
    },
    "use_boundary_conditions": True,
}

# Create the model
small_world = SpeciesModel(**model_params)

SolaraViz(
    small_world,
    components=[
        # make_space_component(agent_portrayal=agent_draw, backend="matplotlib"),
        SpaceWithSensing
    ],
    model_params=model_params,
    name="Predator-Prey Simulation",
)

### Static checks (IGNORE)

In [ ]:
def plot_headings(positions, headings, sensing_radius, animal, old_positions=np.array([]), old_headings=np.array([])):
    # Plot 
    fig, ax = plt.subplots(1,1)
    # Plot deer
    ax.scatter(deer_positions[:,0], deer_positions[:,1], c='blue', label='deer')
    # Plot wolves
    ax.scatter(positions[:,0], positions[:,1], c='red', label='wolf')

    # Plot headings
    arrow_scale = 1000    
    ax.quiver(
        positions[:,0], 
        positions[:,1], 
        headings[:,0] * arrow_scale, 
        headings[:,1] * arrow_scale, 
        width=0.003,                # Arrow shaft width
        headwidth=4,                # Arrow head width
        headlength=5,
        label='heading vector')
    
    # plot sensing radius
    for i in range(len(positions)):
        circ = plt.Circle((positions[i][0], positions[i][1]), radius=sensing_radius, edgecolor='b', facecolor='None')
        ax.add_patch(circ) 

    if len(old_headings) > 0 and len(old_positions) > 0:
        # Plot positions
        ax.scatter(old_positions[:,0], old_positions[:,1], c='red', label='previous position', alpha=0.4)
        
        ax.quiver(
            old_positions[:,0], 
            old_positions[:,1], 
            old_headings[:,0] * arrow_scale, 
            old_headings[:,1] * arrow_scale, 
            width=0.003,                # Arrow shaft width
            headwidth=4,                # Arrow head width
            headlength=5,
            alpha=0.4, 
            color='black',
            label='previous headings'
            )

    

    ax.set_title(f'Demonstrating {animal} movement')
    ax.legend(loc='upper right')
    plt.show()

In [ ]:
num_wolves = 5
num_deer = 5

# Create smaller world 
small_world = SpeciesModel(
    init_predators=num_wolves,
    init_deer = num_deer,
    height=30000,     
    width=30000,
    seed=40,
    init_num_of_packs = 3,
    predator = 'Wolf',  # Helper attribute to avoid imports when accessing agent type
    energy_decrease = 0.05,  # Energy decrease parameter 
    energy_min = 0,  # Point at which the animal will die of exhaustion
    veg_cell_size = 1,  # Introducing vegetation
    # Options to control complexity of the model
    use_pack_dynamics = True,  
    use_random_movement = False,
    use_veg = False
)

wolf_positions = []
deer_positions = []
wolves = []
deer = []
for agent in small_world.space.agents:
    x,y = agent.pos
    # check if wolf
    if agent.species == 'Wolf':
        wolf_positions.append([x,y])   
        wolves.append(agent)     
    elif agent.species == 'Deer':
        deer_positions.append([x,y])
        deer.append(agent)     

deer_positions = np.array(deer_positions)

wolf_headings = np.zeros((num_wolves,2))
for i,w in enumerate(wolves):
    wolf_headings[i] = w.heading


# Perform some steps
n_steps = 5
headings = np.zeros((n_steps,len(wolf_headings), 2))
positions = np.zeros((n_steps, len(wolf_positions),2))

for step in range(n_steps):
    # Move the wolves each step, storing their positions and headings
    for i,w in enumerate(wolves):
        w.move()
        headings[step][i] = w.heading
        x,y = w.pos
        positions[step][i] = [x,y]



plot_headings(positions[2], headings[2], wolves[0].sensing_radius, 'Wolf', positions[1], headings[1])
plot_headings(positions[4], headings[4], wolves[0].sensing_radius, 'Wolf', positions[3], headings[3])

